## Steps

1. Data Loading
2. Train-Test-Split
3. Preprocessing Pipeline
4. Model training and evaluation
5. Streamlit Deployment

### 1. Data Loading

In [ ]:
!pip -q install gdown

import gdown
import pandas as pd

file_id = "1quWEMNIzcr_ZgKl9qEQNeippnmE6csiz"
url = f"https://drive.google.com/uc?id={file_id}"

output = "/content/network_traffic_cleaned.csv"
gdown.download(url, output, quiet=False)

df = pd.read_csv(output)

print("Dataset shape:", df.shape)
display(df.head())

Downloading...
From: https://drive.google.com/uc?id=1quWEMNIzcr_ZgKl9qEQNeippnmE6csiz
To: /content/network_traffic_cleaned.csv
100%|██████████| 123k/123k [00:00<00:00, 5.05MB/s]

Dataset shape: (1000, 17)


,packet_size,inter_arrival_time,src_port,dst_port,packet_count_5s,spectral_entropy,frequency_band_energy,label,protocol_type_TCP,protocol_type_UDP,src_ip_192.168.1.2,src_ip_192.168.1.3,dst_ip_192.168.1.5,dst_ip_192.168.1.6,tcp_flags_FIN,tcp_flags_SYN,tcp_flags_SYN-ACK
0,0.405154,0.620362,62569,443,0.857143,0.834066,0.534891,0.0,0,1,1,0,0,0,0,0,0
1,0.527559,0.741288,59382,443,0.785714,0.147196,0.990757,0.0,0,1,0,0,0,1,0,1,0
2,0.226199,0.485116,65484,80,0.285714,0.855192,0.031781,0.0,0,1,0,0,1,0,0,0,0
3,0.573372,0.450965,51707,53,0.142857,0.153220,0.169958,0.0,0,0,0,1,0,0,0,0,0
4,0.651396,0.888740,26915,53,0.714286,0.923916,0.552053,0.0,1,0,0,1,0,0,0,1,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   packet_size            1000 non-null   float64
 1   inter_arrival_time     1000 non-null   float64
 2   src_port               1000 non-null   int64  
 3   dst_port               1000 non-null   int64  
 4   packet_count_5s        1000 non-null   float64
 5   spectral_entropy       1000 non-null   float64
 6   frequency_band_energy  1000 non-null   float64
 7   label                  1000 non-null   float64
 8   protocol_type_TCP      1000 non-null   int64  
 9   protocol_type_UDP      1000 non-null   int64  
 10  src_ip_192.168.1.2     1000 non-null   int64  
 11  src_ip_192.168.1.3     1000 non-null   int64  
 12  dst_ip_192.168.1.5     1000 non-null   int64  
 13  dst_ip_192.168.1.6     1000 non-null   int64  
 14  tcp_flags_FIN          1000 non-null   int64  
 15  tcp_f

### 2. Train-Test-Split

In [ ]:
# Separate input features and target
X = df.drop(columns=['label']).copy()
y = df['label'].astype("int8").copy()

print("Feature matrix shape:",X.shape)
print("Target shape:",y.shape)

print("\nTarget Distribution:")
print(y.value_counts().sort_index())

Feature matrix shape: (1000, 16)
Target shape: (1000,)

Target Distribution:
label
0    900
1    100
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Training set:",X_train.shape,y_train.shape)
print("Testing set:",X_test.shape,y_test.shape)

Training set: (800, 16) (800,)
Testing set: (200, 16) (200,)


- `test_size=0.20`: 800 training records and 200 test records.
- `random_state=42`: makes the split reproducible
- `stratify=y`: preserves the class proportions in both sets.

### 3. Preprocessing Pipeline

- To convert `dst_port` categorical value into numerical we use `OnehotEncoding`
- `src_port` has 995 distinct values; one-hot encoding could create too many columns.
- Excluding `src_port` from the initial baseline, using onehot for `dst_port` because it has only three categories.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


categorical_features = ['dst_port']

excluded_features = ['src_port','dst_port']

numeric_features = [
    col for col in X_train.columns
    if col not in excluded_features
]

preprocessor = ColumnTransformer([
    ('categorical',OneHotEncoder(handle_unknown='ignore'),categorical_features),
    ('numeric',"passthrough",numeric_features)
])




In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:",X_train.shape)
print("Processed training shape:",X_train_processed.shape)

print("\nOriginal testing shape:",X_test.shape)
print("Processed testing shape:",X_test_processed.shape)

Original training shape: (800, 16)
Processed training shape: (800, 17)

Original testing shape: (200, 16)
Processed testing shape: (200, 17)


In [ ]:
# Get the names of the transformed features
feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("\nTransformed feature names:")

for name in feature_names:
    print(name)

Number of transformed features: 17

Transformed feature names:
categorical__dst_port_53
categorical__dst_port_80
categorical__dst_port_443
numeric__packet_size
numeric__inter_arrival_time
numeric__packet_count_5s
numeric__spectral_entropy
numeric__frequency_band_energy
numeric__protocol_type_TCP
numeric__protocol_type_UDP
numeric__src_ip_192.168.1.2
numeric__src_ip_192.168.1.3
numeric__dst_ip_192.168.1.5
numeric__dst_ip_192.168.1.6
numeric__tcp_flags_FIN
numeric__tcp_flags_SYN
numeric__tcp_flags_SYN-ACK


### 4. Model training and evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ))
])

In [ ]:
model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['dst_port']),
                                                 ('numeric', 'passthrough',
                                                  ['packet_size',
                                                   'inter_arrival_time',
                                                   'packet_count_5s',
                                                   'spectral_entropy',
                                                   'frequency_band_energy',
                                                   'protocol_type_TCP',
                                                   'protocol_type_UDP',
                                                   'src_ip_192.168.1.2',
                                                   'src_ip_192.168.1.3',
                                                   'dst_ip_192.168.1.5',
                                                   'dst_ip_192.168.1.6',
                                                   'tcp_flags_FIN',
                                                   'tcp_flags_SYN',
                                                   'tcp_flags_SYN-ACK'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model_pipeline.predict(X_test)

print("Accuracy:",round(accuracy_score(y_test,y_pred),4))
print("\nClassification Report:")
print(classification_report(
    y_test,y_pred,
    target_names=["Normal","Malicious"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test,y_pred))


Accuracy: 0.9

Classification Report:
              precision    recall  f1-score   support

      Normal       0.90      1.00      0.95       180
   Malicious       0.00      0.00      0.00        20

    accuracy                           0.90       200
   macro avg       0.45      0.50      0.47       200
weighted avg       0.81      0.90      0.85       200


Confusion Matrix:
[[180   0]
 [ 20   0]]


- Our RandomForest achieved `90 %` accuracy, but it failed to detect every malicious record.
- Malicious records detected: `0/20`

In [ ]:
# Inspect the model's predicted probabilities

import pandas as pd

y_proba = model_pipeline.predict_proba(X_test)
malicious_proba = y_proba[:,1]

print("Malicious probability summary:")
print(pd.Series(malicious_proba).describe())

print("\nHighest malicious probabilities:")
print(
    pd.Series(malicious_proba)
    .sort_values(ascending=False)
    .head(10)
    .to_list()
)

print("\nMaximum malicious probability:",
      round(malicious_proba.max(), 4))

Malicious probability summary:
count    200.000000
mean       0.091550
std        0.058705
min        0.000000
25%        0.050000
50%        0.090000
75%        0.130000
max        0.270000
dtype: float64

Highest malicious probabilities:
[0.27, 0.26, 0.25, 0.24, 0.23, 0.22, 0.21, 0.21, 0.21, 0.21]

Maximum malicious probability: 0.27


In [ ]:
# Checking whether the model learned anything from the training data

y_train_pred = model_pipeline.predict(X_train)

print("Training Classification Report:")
print(
    classification_report(
        y_train,
        y_train_pred,
        target_names=["Normal", "Malicious"],
        zero_division=0
    )
)

print("Training Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred))

Training Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       720
   Malicious       1.00      1.00      1.00        80

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800

Training Confusion Matrix:
[[720   0]
 [  0  80]]


- Model perfectly memorizes the training examples, but fails on the held-out examples. This is consistent iwth overfitting.